# Step 2b - Does letter order matter? Markov chains, AR and ARMA (Ilgaz)

**Question:** two sequences can have the same letter percentages and still be different "texts" (`CG` ≠ `GC`). How much of the promoter signal is **composition** (which letters), how much is **local order** (which letters sit next to each other), and how much needs **position** (where, relative to the TSS)?

**Ladder of models in this notebook (all classical, laptop-fast):**

| Level | Model | Sees |
|---|---|---|
| 0 | Shuffle control, anagram k-mers | Composition vs. order, without a model |
| 1 | **Markov chain** of order k (one per class) | The next letter given the previous k letters: the categorical "AR model" |
| 2 | **AR(p)** on numeric encodings | Letters → numbers (GC = 1/0, purine = 1/0, EIIP), then classic autoregression |
| 3 | **ARMA(p, q)** (statsmodels maximum likelihood) | AR plus moving-average terms |

None of these models sees **position**. That's the job of the BiLSTM / DNABERT-2 (Alejandro's steps 4–6).

**Rules:** fit on train, choose settings on **dev only**, score **test once** for the chosen model of each family. Main metric: MCC.

In [ ]:
import os, time, warnings
from collections import Counter
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("..") / "data" / "prom_300_all"
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
PROM, NON, GREY = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#ecebe8", "figure.dpi": 110, "axes.axisbelow": True})
train, dev, test = [pd.read_csv(DATA_DIR / f"{s}.csv") for s in ("train", "dev", "test")]
N_JOBS = max(1, (os.cpu_count() or 2) - 1)

def scores(y, p):
    yh = (np.asarray(p) >= 0.5).astype(int)
    return {"mcc": matthews_corrcoef(y, yh), "accuracy": accuracy_score(y, yh), "f1": f1_score(y, yh), "roc_auc": roc_auc_score(y, p)}

def save(fig, name, note=None):
    fig.tight_layout(rect=(0, 0.07 if note else 0, 1, 0.9 if fig.legends else 1))
    if note: fig.text(0.5, 0.012, note, ha="center", va="bottom", fontsize=9, style="italic", color="#3d3c39")
    fig.savefig(RESULTS_DIR / name, dpi=150, bbox_inches="tight")

RESULTS = []   # one row per (family, setting, split)
def record(family, setting, split, y, p, seconds=None, n_params=None):
    row = {"family": family, "setting": setting, "split": split, **{k: round(v, 4) for k, v in scores(y, p).items()},
           "seconds": None if seconds is None else round(seconds, 1), "n_params": n_params}
    RESULTS.append(row); return row
print(f"train {len(train)}, dev {len(dev)}, test {len(test)}; parallel jobs: {N_JOBS}")

## 0. Composition vs. order, without a model
**Shuffle control:** shuffle the letters *inside* every sequence. That keeps each sequence's exact letter percentages and destroys its order. Then train the same k-mer logistic regression on real vs. shuffled sequences. Whatever survives shuffling is composition; what's lost is order.

In [ ]:
rng = np.random.default_rng(0)
def shuffle(s): a = np.array(list(s)); rng.shuffle(a); return "".join(a)
tr_sh, dv_sh = train.sequence.map(shuffle), dev.sequence.map(shuffle)
rows = []
for k in (1, 2, 3, 4, 6):
    for name, a, b in (("real order", train.sequence, dev.sequence), ("letters shuffled", tr_sh, dv_sh)):
        vec = CountVectorizer(analyzer="char", ngram_range=(k, k), lowercase=False)
        clf = LogisticRegression(C=0.1, max_iter=3000).fit(vec.fit_transform(a), train.label)
        rows.append({"k": k, "input": name, "dev MCC": matthews_corrcoef(dev.label, clf.predict(vec.transform(b)))})
shuf = pd.DataFrame(rows).pivot(index="k", columns="input", values="dev MCC").round(3)
shuf["lost to shuffling"] = shuf["real order"] - shuf["letters shuffled"]
shuf

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
x = np.arange(len(shuf)); w = 0.38
b1 = ax.bar(x - w/2, shuf["real order"], w, color=PROM, label="real order")
b2 = ax.bar(x + w/2, shuf["letters shuffled"], w, color=GREY, label="letters shuffled (same %)")
for bars in (b1, b2): ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
ax.set(xticks=x, xticklabels=[f"k={k}" for k in shuf.index], ylabel="dev MCC", title="k-mer logistic regression: real vs shuffled letters", ylim=(0, 0.9))
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
save(fig, "order_01_shuffle_control.png", "Same letter percentages, order destroyed: MCC falls to ~0.35 for every k. Order (not composition) carries most of the signal.")

**Anagram k-mers:** k-mers with the same letters in a different order (e.g. `CG` vs `GC`, `ATG` vs `AGT` vs `GTA`). If composition were all that mattered, anagrams would be equally enriched in promoters.

In [ ]:
def kcount(seqs, k):
    c = Counter()
    for s in seqs: c.update(s[i:i+k] for i in range(len(s) - k + 1))
    return c
tables = {}
for k in (2, 3):
    c1, c0 = kcount(train.sequence[train.label == 1], k), kcount(train.sequence[train.label == 0], k)
    n1, n0 = sum(c1.values()), sum(c0.values())
    lfc = pd.Series({m: np.log2(((c1[m] + 1) / n1) / ((c0[m] + 1) / n0)) for m in map("".join, product("ACGT", repeat=k))})
    groups = lfc.groupby(lfc.index.map(lambda m: "".join(sorted(m))))
    spread = groups.agg(lambda g: g.max() - g.min()).sort_values(ascending=False)
    tables[k] = (lfc, spread)
    print(f"k={k}: anagram groups with the largest spread of promoter enrichment (log2 fold change)")
    for key in spread.index[:4]:
        g = lfc[[m for m in lfc.index if "".join(sorted(m)) == key]].sort_values()
        print("   ", "  ".join(f"{m} {v:+.2f}" for m, v in g.items()))

In [ ]:
lfc3, spread3 = tables[3]
top = spread3.index[:8]
fig, ax = plt.subplots(figsize=(10, 4))
for i, key in enumerate(top):
    g = lfc3[[m for m in lfc3.index if "".join(sorted(m)) == key]]
    ax.scatter([i] * len(g), g.values, color=[PROM if v > 0 else NON for v in g.values], s=30, zorder=3)
    ax.vlines(i, g.min(), g.max(), color=GREY, lw=1)
    for m, v in g.items(): ax.text(i + 0.08, v, m, fontsize=7, va="center")
ax.axhline(0, color=GREY, lw=1)
ax.set(xticks=range(len(top)), xticklabels=[f"{{{','.join(sorted(set(k)))}}}\n{k}" for k in top], ylabel="log2 enrichment\n(promoter / non)",
       title="Same letters, different order: 3-mer anagram groups")
save(fig, "order_02_anagrams.png", "Anagrams can point in opposite directions (ACG −0.66 vs CAG +0.35): the CG pair, not the letters, carries the signal.")

## 1. Markov chain classifier
A **Markov chain of order k** models P(next letter | previous k letters). Order 0 = letter percentages only, order 1 = letter pairs, and so on. It's the categorical version of an autoregressive model, and the classic way CpG islands are detected (Durbin et al., *Biological Sequence Analysis*, ch. 3).

Fit one chain per class on train (add-one smoothing) and classify by the log-likelihood ratio:

score(x) = Σ_i log P₁(x_i | x_{i−k..i−1}) − log P₀(x_i | x_{i−k..i−1})

Promoter if score > 0, i.e. a naive Bayes over overlapping (k+1)-mers. The number of parameters per class is 4^k × 3.

In [ ]:
IDX = {b: i for i, b in enumerate("ACGT")}
def encode_int(s):   # A,C,G,T -> 0..3; anything else -> -1
    return np.array([IDX.get(c, -1) for c in s], dtype=np.int64)
Xtr_i, Xdv_i, Xte_i = [np.stack(d.sequence.map(encode_int).values) for d in (train, dev, test)]

def kmer_ids(X, k):
    """Integer id of every (k+1)-mer window, -1 where the window contains an unknown letter."""
    n, L = X.shape; w = k + 1
    ids = np.zeros((n, L - w + 1), dtype=np.int64); bad = np.zeros_like(ids, dtype=bool)
    for j in range(w):
        col = X[:, j:L - w + 1 + j]; ids = ids * 4 + np.clip(col, 0, 3); bad |= col < 0
    ids[bad] = -1; return ids

def fit_markov(X, k, alpha=1.0):
    ids = kmer_ids(X, k); ids = ids[ids >= 0]
    counts = np.bincount(ids, minlength=4 ** (k + 1)).reshape(-1, 4) + alpha   # rows = context, cols = next letter
    return np.log(counts / counts.sum(1, keepdims=True)).ravel()

def markov_score(X, k, lp1, lp0):
    ids = kmer_ids(X, k); diff = (lp1 - lp0)[np.clip(ids, 0, None)]; diff[ids < 0] = 0
    return diff.sum(1)

sig = lambda z: 1 / (1 + np.exp(-np.clip(z, -50, 50)))
markov = {}
for k in range(0, 9):
    t0 = time.time()
    lp1, lp0 = fit_markov(Xtr_i[train.label.values == 1], k), fit_markov(Xtr_i[train.label.values == 0], k)
    secs = time.time() - t0
    markov[k] = (lp1, lp0)
    record("Markov chain", f"order {k}", "dev", dev.label, sig(markov_score(Xdv_i, k, lp1, lp0)), secs, 2 * 3 * 4 ** k)
mk = pd.DataFrame([r for r in RESULTS if r["family"] == "Markov chain"])
mk

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(range(len(mk)), mk.mcc, marker="o", color=PROM)
for i, v in enumerate(mk.mcc): ax.text(i, v + 0.012, f"{v:.3f}", ha="center", fontsize=8)
ax.set(xticks=range(len(mk)), xticklabels=mk.setting.str.replace("order ", ""), xlabel="Markov order k (letters of context)",
       ylabel="dev MCC", title="Markov chain classifier: more context helps, then overfits", ylim=(0.25, 0.85))
best_k = int(mk.loc[mk.mcc.idxmax(), "setting"].split()[-1])
ax.axvline(best_k, color=GREY, ls="--", lw=1)
save(fig, "order_03_markov_orders.png", f"Order 0 = composition only (MCC {mk.mcc.iloc[0]:.2f}); best is order {best_k} (MCC {mk.mcc.max():.3f}); beyond it there are too few examples per context.")
print("chosen on dev: order", best_k)

In [ ]:
# What does the best chain "say"? Context-specific probabilities: e.g. P(G | C) = the CpG rule
lp1, lp0 = markov[1]
P1, P0 = np.exp(lp1).reshape(4, 4), np.exp(lp0).reshape(4, 4)
tab = pd.DataFrame({f"P(next | {a}) promoter": P1[IDX[a]] for a in "ACGT"} | {f"P(next | {a}) non": P0[IDX[a]] for a in "ACGT"}, index=list("ACGT")).T
print("Order-1 transition probabilities (rows = previous letter and class, columns = next letter):")
tab.round(3)

## 2. AR(p) on numeric encodings
ARMA needs **numbers**, so first we choose an encoding. Each one keeps one physical property of DNA:

| Encoding | A | C | G | T | Meaning |
|---|---|---|---|---|---|
| GC (strong/weak) | 0 | 1 | 1 | 0 | 3 vs 2 hydrogen bonds; GC content |
| purine/pyrimidine | 1 | 0 | 1 | 0 | ring size (A, G = purines) |
| amino/keto | 1 | 1 | 0 | 0 | chemical group (the third binary split) |
| EIIP | 0.1260 | 0.1340 | 0.0806 | 0.1335 | electron-ion interaction potential (genomic signal processing) |

Per sequence: centre it, then fit **AR(p)** by least squares: x_t = φ₁x_{t−1} + … + φ_p x_{t−p} + ε_t.

**Features:**
- the φ's (**order**) only, or
- the φ's + mean + residual variance (**order + composition**). The mean of a 0/1 series *is* its GC %, so this variant mixes the two effects.

A logistic regression then classifies. p and the encoding are chosen on dev.

In [ ]:
ENC = {"GC": {"A": 0, "C": 1, "G": 1, "T": 0}, "purine": {"A": 1, "C": 0, "G": 1, "T": 0},
       "amino": {"A": 1, "C": 1, "G": 0, "T": 0}, "EIIP": {"A": 0.1260, "C": 0.1340, "G": 0.0806, "T": 0.1335}}
def to_series(X, enc):
    lut = np.array([enc[b] for b in "ACGT"] + [np.mean(list(enc.values()))])   # unknown letter -> mean value
    return lut[np.where(X < 0, 4, X)].astype(float)

def ar_features(S, p):
    """Least-squares AR(p) per row of S [n, L]. Returns phi [n, p], mean [n], residual variance [n]."""
    mu = S.mean(1, keepdims=True); Z = S - mu
    n, L = Z.shape
    lagged = np.stack([Z[:, p - j - 1:L - j - 1] for j in range(p)], axis=2)   # [n, L-p, p]
    target = Z[:, p:]
    XtX = np.einsum("ntp,ntq->npq", lagged, lagged) + 1e-8 * np.eye(p)
    Xty = np.einsum("ntp,nt->np", lagged, target)
    phi = np.linalg.solve(XtX, Xty[..., None])[..., 0]
    resid = target - np.einsum("ntp,np->nt", lagged, phi)
    return phi, mu[:, 0], resid.var(1)

def ar_design(X, enc, p, with_composition):
    phi, mu, s2 = ar_features(to_series(X, ENC[enc]), p)
    return np.column_stack([phi, mu, s2]) if with_composition else phi

for enc in ENC:
    for p in (1, 2, 3, 5, 10, 20):
        for comp in (False, True):
            t0 = time.time()
            clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(ar_design(Xtr_i, enc, p, comp), train.label)
            pr = clf.predict_proba(ar_design(Xdv_i, enc, p, comp))[:, 1]
            record("AR(p)", f"{enc}, p={p}, {'order + composition' if comp else 'order only'}", "dev", dev.label, pr, time.time() - t0, p + (2 if comp else 0) + 1)
ar = pd.DataFrame([r for r in RESULTS if r["family"] == "AR(p)"])
ar[["enc", "p", "features"]] = ar.setting.str.extract(r"(\w+), p=(\d+), (.+)"); ar.p = ar.p.astype(int)
ar.pivot_table(index=["enc", "features"], columns="p", values="mcc").round(3)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
colors = {"GC": PROM, "purine": "#3a9d5d", "amino": "#e8a33d", "EIIP": "#7b5ea7"}
for a, feat in zip(axs, ["order only", "order + composition"]):
    for enc, c in colors.items():
        g = ar[(ar.enc == enc) & (ar.features == feat)].sort_values("p")
        a.plot(g.p, g.mcc, marker="o", color=c, label=enc)
    a.set(xscale="log", xticks=[1, 2, 3, 5, 10, 20], xticklabels=[1, 2, 3, 5, 10, 20], xlabel="AR order p", title=f"AR(p) features: {feat}")
axs[0].set_ylabel("dev MCC")
h, l = axs[0].get_legend_handles_labels(); fig.legend(h, l, loc="upper center", ncol=4, frameon=False, title="encoding")
best_ar = ar.loc[ar.mcc.idxmax()]
save(fig, "order_04_ar_grid.png", f"Best AR: {best_ar.setting} (dev MCC {best_ar.mcc:.3f}). Order-only AR coefficients carry far less than the composition terms.")
print("chosen on dev:", best_ar.setting, "MCC", best_ar.mcc)

## 3. ARMA(p, q) by maximum likelihood
A **moving-average** part adds dependence on past *errors*: x_t = Σ φ_i x_{t−i} + ε_t + Σ θ_j ε_{t−j}. We fit real ARMA models with `statsmodels` (exact Gaussian likelihood) on the encoding that won in Section 2.

It costs ~25 ms per sequence, so: **all of dev and test**, and a **stratified 8,000-row train sample** (8,000 of 47,356), in parallel.

Features per sequence: φ's, θ's, mean (constant) and σ². Then logistic regression, as in Section 2.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
def arma_params(series, p, q):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            r = ARIMA(series, order=(p, 0, q), trend="c").fit()
            return np.r_[r.params, float(r.mle_retvals.get("converged", True))]
        except Exception:
            return np.r_[np.full(p + q + 2, np.nan), 0.0]

def arma_design(X, enc, p, q):
    S = to_series(X, ENC[enc])
    out = Parallel(n_jobs=N_JOBS, batch_size=64)(delayed(arma_params)(s, p, q) for s in S)
    out = np.array(out); conv = out[:, -1]; F = out[:, :-1]
    F = np.where(np.isnan(F), np.nanmedian(F, axis=0), F)
    return F, conv     # columns: const, ar.L1.., ma.L1.., sigma2

ENC_ARMA = best_ar.enc
tr_idx = train.groupby("label").sample(4000, random_state=0).index
Xtr_s, ytr_s = Xtr_i[tr_idx], train.label.values[tr_idx]
arma_cache = {}
for (p, q) in [(1, 1), (2, 1), (2, 2)]:
    t0 = time.time()
    Ftr, ctr = arma_design(Xtr_s, ENC_ARMA, p, q); Fdv, cdv = arma_design(Xdv_i, ENC_ARMA, p, q)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(Ftr, ytr_s)
    secs = time.time() - t0
    arma_cache[(p, q)] = clf
    row = record("ARMA(p,q)", f"{ENC_ARMA}, ARMA({p},{q})", "dev", dev.label, clf.predict_proba(Fdv)[:, 1], secs, p + q + 2 + 1)
    print(f"ARMA({p},{q}) on {ENC_ARMA}: dev MCC {row['mcc']:.3f} | converged: train {ctr.mean():.1%}, dev {cdv.mean():.1%} | {secs:.0f}s")

In [ ]:
# Fair comparison: the best AR(p) refit on the SAME 8,000-row train sample
bp, bcomp = int(best_ar.p), best_ar.features == "order + composition"
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(ar_design(Xtr_s, ENC_ARMA, bp, bcomp), ytr_s)
row = record("AR(p) [8k train sample]", f"{best_ar.setting}", "dev", dev.label, clf.predict_proba(ar_design(Xdv_i, ENC_ARMA, bp, bcomp))[:, 1])
print("AR on the same 8k sample: dev MCC", row["mcc"])
am = pd.DataFrame([r for r in RESULTS if r["family"].startswith("ARMA")]); am

**What the moving-average terms add:** compare the ARMA rows with the AR row fitted on the same 8k sample. If MA terms barely help, the dependence is short-range and already captured by a few AR lags, which matches the Markov result: the information is in *which neighbours*, not in long smooth trends. DNA isn't a smooth signal, so ARMA's Gaussian assumptions are a stretch on 0/1 data.

## 4. Comparison and the one test run
Chosen on dev: the best Markov order, the best AR(p) setting and the best ARMA(p,q). Each is scored on **test once**. For context we add the 6-mer TF-IDF logistic regression from `step2_logreg.ipynb` (dev grid in `results/logreg_dev_grid.csv`) and Alejandro's sequence models (from his `Tuning_and_test_findings.md`; not re-run here).

In [ ]:
# Markov: best order on dev -> test
lp1, lp0 = markov[best_k]
record("Markov chain", f"order {best_k}", "test", test.label, sig(markov_score(Xte_i, best_k, lp1, lp0)), n_params=2 * 3 * 4 ** best_k)
# AR: best setting on dev, refit on full train -> test
enc, comp = best_ar.enc, best_ar.features == "order + composition"
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(ar_design(Xtr_i, enc, bp, comp), train.label)
record("AR(p)", best_ar.setting, "test", test.label, clf.predict_proba(ar_design(Xte_i, enc, bp, comp))[:, 1], n_params=bp + (2 if comp else 0) + 1)
# ARMA: best on dev -> test
best_arma = am.loc[am.mcc.idxmax()]
pq = tuple(int(v) for v in best_arma.setting.split("ARMA(")[1].rstrip(")").split(","))
Fte, cte = arma_design(Xte_i, ENC_ARMA, *pq)
record("ARMA(p,q)", best_arma.setting, "test", test.label, arma_cache[pq].predict_proba(Fte)[:, 1], n_params=sum(pq) + 3)
# Order-0 Markov = composition-only reference, on test
lp1, lp0 = markov[0]
record("Markov chain", "order 0", "test", test.label, sig(markov_score(Xte_i, 0, lp1, lp0)), n_params=6)

res = pd.DataFrame(RESULTS); res.to_csv(RESULTS_DIR / "order_models_results.csv", index=False)
final = res[res.split == "test"].copy()
dev_mcc = {(r.family, r.setting): r.mcc for r in res[res.split == "dev"].itertuples()}
final.insert(3, "dev mcc", [dev_mcc.get((f, s)) for f, s in zip(final.family, final.setting)])
final[["family", "setting", "dev mcc", "mcc", "accuracy", "f1", "roc_auc", "n_params"]].rename(columns={"mcc": "test mcc"})

In [ ]:
grid = pd.read_csv(RESULTS_DIR / "logreg_dev_grid.csv") if (RESULTS_DIR / "logreg_dev_grid.csv").exists() else None
ladder = [("Composition only\n(Markov order 0)", final.query("setting == 'order 0'").mcc.iloc[0], GREY),
          ("ARMA" + best_arma.setting.split("ARMA")[1], final.query("family == 'ARMA(p,q)'").mcc.iloc[0], "#7b5ea7"),
          (f"AR(p)\n{best_ar.setting.split(',')[0]}, p={bp}", final.query("family == 'AR(p)'").mcc.iloc[0], "#7b5ea7"),
          (f"Markov chain\norder {best_k}", final.query("family == 'Markov chain' and setting != 'order 0'").mcc.iloc[0], PROM),
          ("6-mer TF-IDF LogReg\n(step 2)", 0.755, PROM),
          ("BiLSTM, single letters\n(Alejandro)", 0.826, "#1f2a44"),
          ("DNABERT-2 fine-tuned\n(Alejandro)", 0.868, "#1f2a44")]
fig, ax = plt.subplots(figsize=(11, 4.2))
bars = ax.bar(range(len(ladder)), [v for _, v, _ in ladder], color=[c for *_, c in ladder])
ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)
ax.set(xticks=range(len(ladder)), xticklabels=[n for n, *_ in ladder], ylabel="test MCC", ylim=(0, 1),
       title="Test MCC ladder: composition → local order → position")
for t in ax.get_xticklabels(): t.set_fontsize(8)
import matplotlib.patches as mpatches
fig.legend(handles=[mpatches.Patch(color=GREY, label="composition"), mpatches.Patch(color="#7b5ea7", label="numeric time series (AR/ARMA)"),
                    mpatches.Patch(color=PROM, label="local order (k-mers / Markov)"), mpatches.Patch(color="#1f2a44", label="order + position (neural)")],
           loc="upper center", ncol=4, frameon=False, fontsize=8)
save(fig, "order_05_test_ladder.png", "Each step up the ladder adds information: local letter order roughly doubles composition's MCC; position-aware neural models add the rest.")

## 5. Conclusions
Numbers below are from this run; see the tables above and `results/order_models_results.csv`.

1. **Order matters more than percentage.** Shuffling letters (same %) drops a 6-mer model from ~0.76 to ~0.35 dev MCC, and anagram k-mers can point in opposite directions (`ACG` vs `CAG`).
2. **Markov chains are the right "time-series" model for DNA.** They work directly on letters, and every extra letter of context helps up to an optimum (chosen on dev), where they come close to the k-mer logistic regression. That's expected, since both model overlapping k-mers.
3. **AR/ARMA on numeric encodings work, but less well.** Collapsing 4 letters to one 0/1 number throws information away (GC encoding can't tell `CG` from `GC`). The order-only AR coefficients are weak, and most of AR's score comes from the mean/variance terms, i.e. composition. The MA terms add little over AR.
4. **None of these see position.** The remaining gap to the BiLSTM (0.826) and DNABERT-2 (0.868) is what position (Inr at 250, TATA region, downstream G surplus) and longer-range patterns are worth.
5. **Caveat from the EDA:** part of every score here is "randomness detection" (the negatives are 60% uniform random DNA). Markov chains are especially good at it: random filler has flat transition probabilities.